In [7]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

C:\Users\Najeeb\AppData\Local\Temp\ipykernel_28400\2246380887.py:7: DeprecationWarning: 'asyncio.get_event_loop_policy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\Najeeb\AppData\Local\Temp\ipykernel_28400\2246380887.py:7: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
C:\Users\Najeeb\AppData\Local\Temp\ipykernel_28400\2246380887.py:8: DeprecationWarning: 'asyncio.WindowsProactorEventLoopPolicy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
C:\Users\Najeeb\AppData\Local\Temp\ipykernel_28400\2246380887.py:8: DeprecationWarning: 'asyncio.set_event_loop_policy' is deprecated and slated for removal in Python 3.16
  asyncio.set_event_loop_policy(

In [1]:
from langchain_ollama.chat_models import ChatOllama

model = ChatOllama(model="qwen3.5")

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_agent": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com"
        }
    }
)

tools = await client.get_tools()

In [4]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [11]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on July 31st 2026")]},
    config
    )

In [12]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st', additional_kwargs={}, response_metadata={}, id='be151a88-fe74-4703-bb83-d52ccf7f7677'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3.5', 'created_at': '2026-04-03T06:23:07.4137848Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8044151500, 'load_duration': 4369399200, 'prompt_eval_count': 1337, 'prompt_eval_duration': 439401700, 'eval_count': 313, 'eval_duration': 3032337000, 'logprobs': None, 'model_name': 'qwen3.5', 'model_provider': 'ollama'}, id='lc_run--019d5202-5b28-77e3-a548-8013f101f48b-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'San Francisco', 'flyTo': 'Tokyo', 'departureDate': '31/03/2024'}, 'id': '336794d8-046f-4e80-a468-4af74f269847', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1337, 'output_tokens': 313, 'total_tokens': 1650}),
              HumanMessage(content

In [ ]:
print(response["messages"][-1].content)